# Day2課題 予測モデル作成のためのデータ作成

In [ ]:
import os

from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import context_precision, context_recall

import pandas as pd

In [ ]:
# API key
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<your-endpoint.openai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "your AzureOpenAI key"

In [ ]:
embedding = AzureOpenAIEmbeddings(model="text-embedding-3-large")
llm = AzureChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
pd_aozora = (
    pd.read_parquet("data/aozora.parquet")
    .query("title.isin(['蜘蛛の糸', 'それから', '銀河鉄道の夜'])")
)
pd_aozora

## 前処理 (主にここを修正)

### チャンキング

In [ ]:
loader = DataFrameLoader(
    pd_aozora[["text"]],
    page_content_column="text",
)
docs = loader.load()
docs

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = recursive_splitter.split_documents(docs)
split_docs[:20]

### VectorStore構築

In [ ]:
vectorstore = InMemoryVectorStore(embedding)
retriever = vectorstore.as_retriever()

# バッチでドキュメントを追加
tmp_list = []
c = 0
limit_c = 200000
for i, doc in enumerate(split_docs):
    c += len(doc.page_content)
    tmp_list.append(doc)
    if c > limit_c or i+1 == len(split_docs):
        print(f"Adding {i+1} documents of {len(split_docs)} to retriever")
        retriever.add_documents(tmp_list)
        tmp_list = []
        c = 0

## RAG構築

In [ ]:
prompt_str = \
"""あなたは質問応答のアシスタントです。質問に応えるために以下の文脈の情報のみを使用してください。答えがわからない場合は「わかりません」と答えてください。
質問: {question}
文脈: {context}
応答: """
prompt = ChatPromptTemplate([("human", prompt_str)])

In [ ]:
rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

## 評価

In [ ]:
test_df = pd.read_parquet("data/aozora_testdata.parquet")
test_df

In [ ]:
user_input = test_df["question"].tolist()
reference = test_df["answer"].tolist()
retrieved_contexts = retriever.batch(user_input)
responce = rag_chain.batch(user_input)

In [ ]:
result_df = pd.DataFrame({
    "user_input": user_input,
    "retrieved_contexts": [[doc.page_content for doc in context] for context in retrieved_contexts],
    "responce": responce,
    "reference": reference,
})
result_df

In [ ]:
result = evaluate(
    EvaluationDataset.from_pandas(result_df),
    llm=LangchainLLMWrapper(llm),
    embeddings=LangchainEmbeddingsWrapper(embedding),
    metrics=[context_precision, context_recall],
)
print(result)

In [ ]:
evaluate_df = pd.concat([result_df, pd.DataFrame(result.scores)], axis="columns")
evaluate_df